In [1]:

from pathlib import Path
import pandas as pd

# Project directories
PROJECT_ROOT = Path.cwd()

# If the notebook is opened from the notebooks folder,
# the project root is its parent directory.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ANNOTATIONS_DIR = PROJECT_ROOT / "data" / "annotations"

# Load the training metadata
annotations = pd.read_csv(
    ANNOTATIONS_DIR / "annotations_train.csv"
)

labels = pd.read_csv(
    ANNOTATIONS_DIR / "image_labels_train.csv"
)

# Inspect dataset dimensions
print("BOUNDING-BOX ANNOTATIONS")
print("Shape:", annotations.shape)
print("Columns:", annotations.columns.tolist())

print("\nCLASSIFICATION LABELS")
print("Shape:", labels.shape)
print("Columns:", labels.columns.tolist())


BOUNDING-BOX ANNOTATIONS
Shape: (69052, 7)
Columns: ['image_id', 'rad_id', 'class_name', 'x_min', 'y_min', 'x_max', 'y_max']

CLASSIFICATION LABELS
Shape: (45000, 30)
Columns: ['image_id', 'rad_id', 'Aortic enlargement', 'Atelectasis', 'Calcification', 'Cardiomegaly', 'Clavicle fracture', 'Consolidation', 'Edema', 'Emphysema', 'Enlarged PA', 'ILD', 'Infiltration', 'Lung Opacity', 'Lung cavity', 'Lung cyst', 'Mediastinal shift', 'Nodule/Mass', 'Pleural effusion', 'Pleural thickening', 'Pneumothorax', 'Pulmonary fibrosis', 'Rib fracture', 'Other lesion', 'COPD', 'Lung tumor', 'Pneumonia', 'Tuberculosis', 'Other diseases', 'No finding']


In [2]:

# Basic dataset statistics
print("Unique training images in annotations:",
      annotations["image_id"].nunique())

print("Unique training images in classification labels:",
      labels["image_id"].nunique())

# Identify rows containing actual bounding boxes
bbox_columns = ["x_min", "y_min", "x_max", "y_max"]

has_bbox = annotations[bbox_columns].notna().all(axis=1)

print("Rows with bounding boxes:", has_bbox.sum())

print("Unique images with bounding boxes:",
      annotations.loc[has_bbox, "image_id"].nunique())

# Count findings by class
print("\nMost frequent annotated findings:")
print(
    annotations.loc[has_bbox, "class_name"]
    .value_counts()
    .head(15)
)


Unique training images in annotations: 15000
Unique training images in classification labels: 15000
Rows with bounding boxes: 37367
Unique images with bounding boxes: 4522

Most frequent annotated findings:
class_name
Aortic enlargement    7193
Cardiomegaly          5443
Pleural thickening    4884
Pulmonary fibrosis    4659
Nodule/Mass           2611
Lung Opacity          2493
Pleural effusion      2483
Other lesion          2228
Infiltration          1247
ILD                   1015
Calcification          967
Consolidation          556
Rib fracture           300
Atelectasis            280
Mediastinal shift      270
Name: count, dtype: int64


In [3]:
# The notebook is inside notebooks/, so go up one level
PROJECT_ROOT = Path.cwd().parent

ANNOTATIONS_PATH = (
    PROJECT_ROOT / "data" / "annotations"
    / "annotations_train.csv"
)

LABELS_PATH = (
    PROJECT_ROOT / "data" / "annotations"
    / "image_labels_train.csv"
)

# Check paths before loading anything
print("Project root:", PROJECT_ROOT)
print("Annotations CSV exists:", ANNOTATIONS_PATH.exists())
print("Labels CSV exists:", LABELS_PATH.exists())

if not ANNOTATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find annotations CSV: {ANNOTATIONS_PATH}"
    )

if not LABELS_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find labels CSV: {LABELS_PATH}"
    )

annotations = pd.read_csv(ANNOTATIONS_PATH)
labels = pd.read_csv(LABELS_PATH)

print("\nBoth CSV files loaded successfully!")
print("Annotations shape:", annotations.shape)
print("Labels shape:", labels.shape)


Project root: c:\Users\johnm\chest-xray-ai-annotation
Annotations CSV exists: True
Labels CSV exists: True

Both CSV files loaded successfully!
Annotations shape: (69052, 7)
Labels shape: (45000, 30)


In [4]:

# Keep only rows with complete bounding-box coordinates
valid_boxes = annotations.dropna(
    subset=["x_min", "y_min", "x_max", "y_max"]
).copy()

# Get unique images that have bounding-box annotations
annotated_image_ids = set(
    valid_boxes["image_id"].unique()
)

# Get images present in the classification labels CSV
label_image_ids = set(
    labels["image_id"].dropna().unique()
)

# Keep images represented in both CSV files
eligible_ids = sorted(
    annotated_image_ids & label_image_ids
)

print("Unique images with valid bounding boxes:",
      len(annotated_image_ids))

print("Images also present in classification labels:",
      len(eligible_ids))

print("\nSample eligible image IDs:")
print(eligible_ids[:10])


Unique images with valid bounding boxes: 4522
Images also present in classification labels: 4522

Sample eligible image IDs:
['0005e8e3701dfb1dd93d53e2ff537b6e', '0007d316f756b3fa0baea2ff514ce945', '000d68e42b71d3eac10ccc077aba07c1', '00150343289f317a0ad5629d5b7d9ef9', '001d127bad87592efe45a5c7678f8b8d', '0021df30f3fddef551eb3df4354b1d06', '0046f681f078851293c4e710c4466058', '0061cf6d35e253b6e7f03940592cc35e', '00675cd546313f912cadd4ad54415d69', '008b3176a7248a0a189b5731ac8d2e95']


In [5]:

# Select 300 unique images reproducibly
import random

random.seed(42)

selected_image_ids = random.sample(eligible_ids, 300)

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

manifest = pd.DataFrame({
    "image_id": selected_image_ids
})

manifest_path = output_dir / "subset_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Selected unique images:", manifest["image_id"].nunique())
print("Manifest saved to:", manifest_path)
print("\nFirst 10 selected image IDs:")
print(manifest.head(10).to_string(index=False))


Selected unique images: 300
Manifest saved to: c:\Users\johnm\chest-xray-ai-annotation\data\processed\subset_manifest.csv

First 10 selected image IDs:
                        image_id
32e735ff23d1c5bf5d2d1aa51937ca82
0b1b897b1e1e170f1b5fd7aeff553afa
80234017eb6caeada9e14d421eb0b8f8
723f93a3f0e3905c498839d23b961631
6833c509c10b569d6167db70b83ee5d5
3f0efef8e4a56b73d2b82e83e50c0b90
2fd3d0e3b5de5535994ac58f2bd175da
fc47195078badf33f2575c3f0db648ab
285cdfec4d725d327266833adf3ba9d5
c32573547f924b6b663798730d915724


In [6]:
manifest = pd.read_csv(manifest_path)

print("Total rows:", len(manifest))
print("Unique image IDs:", manifest["image_id"].nunique())
print("Missing image IDs:", manifest["image_id"].isna().sum())

Total rows: 300
Unique image IDs: 300
Missing image IDs: 0
